## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetische Sequenzaufgabe erstellen ──────────────────────────────────
# Aufgabe: Erkennung ob eine Sequenz mehr positive als negative Zahlen enthält
np.random.seed(42)
SEQUENZ_LAENGE = 30
N_PROBEN       = 3000

X_raw = np.random.randn(N_PROBEN, SEQUENZ_LAENGE).astype("float32")
y_raw = (X_raw.sum(axis=1) > 0).astype("float32")  # 1 wenn Summe positiv

# Zur RNN-Eingabe umformen: (N, T, 1)
X = X_raw[:, :, np.newaxis]
y = y_raw

# Train-/Test-Split
split   = int(0.8 * N_PROBEN)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"Trainingsdaten: {X_train.shape}  Labels: {y_train.shape}")
print(f"Klassen-Balance: {y_train.mean():.2%} positiv")

# ── 2. Einfaches LSTM ─────────────────────────────────────────────────────────
modell_lstm = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=(SEQUENZ_LAENGE, 1), name="lstm"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1,  activation="sigmoid"),
], name="Einfaches_LSTM")

# ── 3. Bidirektionales LSTM ───────────────────────────────────────────────────
modell_bilstm = tf.keras.Sequential([
    # Bidirectional: verarbeitet Sequenz vorwärts UND rückwärts
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(32, name="lstm_kern"),
        input_shape=(SEQUENZ_LAENGE, 1),
        merge_mode="concat",   # beide Richtungen werden zusammengeführt
        name="bi_lstm"
    ),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1,  activation="sigmoid"),
], name="Bidirektionales_LSTM")

# ── 4. Kompilieren ────────────────────────────────────────────────────────────
for m in [modell_lstm, modell_bilstm]:
    m.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

modell_lstm.summary()
modell_bilstm.summary()

print(f"\nEinfaches LSTM    – Parameter: {modell_lstm.count_params():,}")
print(f"Bidirektional LSTM – Parameter: {modell_bilstm.count_params():,}")

# ── 5. Training ───────────────────────────────────────────────────────────────
EPOCHEN = 15

print("\nTrainiere einfaches LSTM...")
h_lstm = modell_lstm.fit(
    X_train, y_train, epochs=EPOCHEN, batch_size=64,
    validation_data=(X_test, y_test), verbose=1
)

print("\nTrainiere bidirektionales LSTM...")
h_bilstm = modell_bilstm.fit(
    X_train, y_train, epochs=EPOCHEN, batch_size=64,
    validation_data=(X_test, y_test), verbose=1
)

# ── 6. Ergebnisse ─────────────────────────────────────────────────────────────
acc_lstm   = modell_lstm.evaluate(X_test,   y_test, verbose=0)[1]
acc_bilstm = modell_bilstm.evaluate(X_test, y_test, verbose=0)[1]

print(f"\nTest-Genauigkeit LSTM:              {acc_lstm:.4f}")
print(f"Test-Genauigkeit Bidirektional:     {acc_bilstm:.4f}")
print(f"Verbesserung:                       {(acc_bilstm - acc_lstm)*100:+.2f}%")
print("\nWarum besser? Bidirektionales LSTM sieht die Sequenz")
print("sowohl von links→rechts als auch von rechts←links.")
print("Für eine Summe über die ganze Sequenz ist das sehr hilfreich!")

# ── 7. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(h_lstm.history["val_accuracy"],
             label="LSTM (Einweg)", linewidth=2)
axes[0].plot(h_bilstm.history["val_accuracy"],
             label="Bidirektional LSTM", linewidth=2)
axes[0].set_title("Validierungs-Genauigkeit")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Genauigkeit")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(h_lstm.history["val_loss"],
             label="LSTM (Einweg)", linewidth=2)
axes[1].plot(h_bilstm.history["val_loss"],
             label="Bidirektional LSTM", linewidth=2)
axes[1].set_title("Validierungs-Verlust")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Verlust")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Bidirektionales LSTM vs. Einfaches LSTM\n"
             "(Sequenz-Klassifikation: Summe positiv?)", fontsize=13)
plt.tight_layout()
plt.savefig("F9_1_bidirektional_lstm.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F9_1_bidirektional_lstm.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetische Aktienkurs-ähnliche Daten generieren ─────────────────────
np.random.seed(42)
N_TAGE = 1000

# Zufälliger Spaziergang + Trend + Saisonalität
trend        = np.linspace(100, 120, N_TAGE)
saisonalitaet = 5 * np.sin(np.linspace(0, 10 * np.pi, N_TAGE))
rauschen     = np.cumsum(np.random.randn(N_TAGE) * 0.5)   # Random Walk
kursdaten    = (trend + saisonalitaet + rauschen).astype("float32")

# Normalisierung auf [0, 1]
kurs_min, kurs_max = kursdaten.min(), kursdaten.max()
kursdaten_norm = (kursdaten - kurs_min) / (kurs_max - kurs_min)

print(f"Kursdaten: {N_TAGE} Handelstage")
print(f"Preisspanne: {kurs_min:.2f} – {kurs_max:.2f}")

# ── 2. Fenster-Datensatz erstellen (multi-step Vorhersage) ────────────────────
FENSTERGRÖSSE = 30   # Blick auf die letzten 30 Tage
VORHERSAGE_SCHRITTE = 5  # Vorhersage der nächsten 5 Tage

def multi_step_fenster(reihe, fenstergr, vorhersage_schritte):
    X, y = [], []
    for i in range(len(reihe) - fenstergr - vorhersage_schritte):
        X.append(reihe[i : i + fenstergr])
        y.append(reihe[i + fenstergr : i + fenstergr + vorhersage_schritte])
    return (np.array(X)[:, :, np.newaxis].astype("float32"),
            np.array(y).astype("float32"))

X, y = multi_step_fenster(kursdaten_norm, FENSTERGRÖSSE, VORHERSAGE_SCHRITTE)
print(f"X: {X.shape}, y: {y.shape}")

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ── 3. Gestapeltes LSTM-Modell (3 Schichten) ──────────────────────────────────
modell = tf.keras.Sequential([
    # Erste LSTM-Schicht: return_sequences=True → gibt Sequenz weiter
    tf.keras.layers.LSTM(64, return_sequences=True,
                         input_shape=(FENSTERGRÖSSE, 1), name="lstm_1"),
    tf.keras.layers.Dropout(0.2),

    # Zweite LSTM-Schicht: return_sequences=True
    tf.keras.layers.LSTM(64, return_sequences=True, name="lstm_2"),
    tf.keras.layers.Dropout(0.2),

    # Dritte LSTM-Schicht: return_sequences=False → gibt nur letzten Schritt aus
    tf.keras.layers.LSTM(32, return_sequences=False, name="lstm_3"),
    tf.keras.layers.Dropout(0.2),

    # Ausgabe: nächste 5 Tage vorhersagen
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(VORHERSAGE_SCHRITTE, name="ausgabe"),
], name="Gestapeltes_LSTM_3_Schichten")

modell.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)
modell.summary()

# ── 4. Training ───────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=5, verbose=1)
]
print("\nTrainiere gestapeltes LSTM (3 Schichten)...")
history = modell.fit(
    X_train, y_train, epochs=60, batch_size=32,
    validation_split=0.1, callbacks=callbacks, verbose=1
)

# ── 5. Vorhersage der nächsten 5 Tage ────────────────────────────────────────
vorhersagen_norm = modell.predict(X_test, verbose=0)

# Rücktransformation auf ursprüngliche Skala
vorhersagen = vorhersagen_norm * (kurs_max - kurs_min) + kurs_min
y_wahr      = y_test           * (kurs_max - kurs_min) + kurs_min

mae = np.mean(np.abs(vorhersagen - y_wahr))
print(f"\nTest-MAE (Originalskala): {mae:.4f}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vorhersagen vs. Tatsächlich (erste 10 Testsequenzen, Tag 1 der Vorhersage)
n_zeigen = 50
axes[0].plot(y_wahr[:n_zeigen, 0],          label="Tatsächlich (Tag+1)", linewidth=2)
axes[0].plot(vorhersagen[:n_zeigen, 0],     label="Vorhersage (Tag+1)",
             linestyle="--", linewidth=2)
axes[0].plot(vorhersagen[:n_zeigen, 4],     label="Vorhersage (Tag+5)",
             linestyle="-.", linewidth=1.5, alpha=0.7)
axes[0].set_title(f"Stacked LSTM: {VORHERSAGE_SCHRITTE}-Tage Vorhersage")
axes[0].set_xlabel("Testsequenz")
axes[0].set_ylabel("Aktienkurs (normalisiert)")
axes[0].legend()
axes[0].grid(True)

# Trainingsverlauf
axes[1].plot(history.history["loss"],     label="Training-MSE")
axes[1].plot(history.history["val_loss"], label="Validierungs-MSE")
axes[1].set_title("Trainingsverlauf")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("MSE")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Stacked LSTM (3 Schichten) für Aktienkurs-Vorhersage", fontsize=13)
plt.tight_layout()
plt.savefig("F9_2_stacked_lstm.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F9_2_stacked_lstm.png")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import time
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Sinuswellen-Zeitreihe ──────────────────────────────────────────────────
np.random.seed(42)
FENSTERGRÖSSE = 25
N_PUNKTE      = 2000

t = np.linspace(0, 6 * np.pi, N_PUNKTE)
zeitreihe = (np.sin(t) + 0.1 * np.random.randn(N_PUNKTE)).astype("float32")

def fenster_erstellen(reihe, fenstergr):
    X, y = [], []
    for i in range(len(reihe) - fenstergr):
        X.append(reihe[i : i + fenstergr])
        y.append(reihe[i + fenstergr])
    return np.array(X)[:, :, np.newaxis], np.array(y)

X, y = fenster_erstellen(zeitreihe, FENSTERGRÖSSE)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ── 2. LSTM-Modell ────────────────────────────────────────────────────────────
def lstm_erstellen():
    return tf.keras.Sequential([
        tf.keras.layers.LSTM(64, input_shape=(FENSTERGRÖSSE, 1), name="lstm"),
        tf.keras.layers.Dense(1),
    ], name="LSTM")

# ── 3. GRU-Modell ─────────────────────────────────────────────────────────────
def gru_erstellen():
    return tf.keras.Sequential([
        # GRU: vereinfachter Mechanismus (Update + Reset Gate statt 4 Gates)
        tf.keras.layers.GRU(64, input_shape=(FENSTERGRÖSSE, 1), name="gru"),
        tf.keras.layers.Dense(1),
    ], name="GRU")

m_lstm = lstm_erstellen()
m_gru  = gru_erstellen()

for m in [m_lstm, m_gru]:
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])

# ── 4. Parametervergleich ──────────────────────────────────────────────────────
print("── Parametervergleich ──")
print(f"LSTM-Parameter: {m_lstm.count_params():,}")
print(f"GRU-Parameter:  {m_gru.count_params():,}")
print("(GRU hat ~75% der LSTM-Parameter: 2 statt 4 Gates)")

# Erklärung
print("\nLSTM Gates: Input, Forget, Output, Cell State (4 × W-Matrizen)")
print("GRU Gates:  Update, Reset           (3 × W-Matrizen, kein separater Cell State)")

# ── 5. Training mit Zeitmessung ───────────────────────────────────────────────
EPOCHEN = 20

print("\nTrainiere LSTM...")
start = time.perf_counter()
h_lstm = m_lstm.fit(X_train, y_train, epochs=EPOCHEN, batch_size=32,
                     validation_data=(X_test, y_test), verbose=1)
zeit_lstm = time.perf_counter() - start

print("\nTrainiere GRU...")
start = time.perf_counter()
h_gru  = m_gru.fit(X_train, y_train, epochs=EPOCHEN, batch_size=32,
                    validation_data=(X_test, y_test), verbose=1)
zeit_gru = time.perf_counter() - start

# ── 6. Ergebnisse ─────────────────────────────────────────────────────────────
mae_lstm = np.mean(np.abs(m_lstm.predict(X_test, verbose=0).flatten() - y_test))
mae_gru  = np.mean(np.abs(m_gru.predict(X_test,  verbose=0).flatten() - y_test))

print("\n── Vergleichstabelle ──")
print(f"{'Modell':<10} {'Parameter':>10} {'Test-MAE':>12} {'Zeit (s)':>10}")
print("-" * 45)
print(f"{'LSTM':<10} {m_lstm.count_params():>10,} {mae_lstm:>12.6f} {zeit_lstm:>10.1f}")
print(f"{'GRU':<10} {m_gru.count_params():>10,}  {mae_gru:>12.6f} {zeit_gru:>10.1f}")
print(f"{'Vorteil':<10} {'GRU ≈25% weniger':>10}  "
      f"{'ähnlich':>12} {'GRU schneller' if zeit_gru < zeit_lstm else 'LSTM schneller':>10}")

# ── 7. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pred_lstm = m_lstm.predict(X_test, verbose=0).flatten()
pred_gru  = m_gru.predict(X_test,  verbose=0).flatten()
n_zeigen  = 100

axes[0].plot(y_test[:n_zeigen],       label="Tatsächlich", linewidth=2)
axes[0].plot(pred_lstm[:n_zeigen],    label=f"LSTM (MAE={mae_lstm:.5f})",
             linestyle="--", linewidth=1.5)
axes[0].plot(pred_gru[:n_zeigen],     label=f"GRU  (MAE={mae_gru:.5f})",
             linestyle="-.", linewidth=1.5)
axes[0].set_title("Vorhersagen: LSTM vs. GRU")
axes[0].set_xlabel("Zeitschritt")
axes[0].legend(fontsize=8)
axes[0].grid(True)

axes[1].plot(h_lstm.history["val_loss"], label="LSTM",  linewidth=2)
axes[1].plot(h_gru.history["val_loss"],  label="GRU",   linewidth=2)
axes[1].set_title("Validierungs-Verlust (MSE)")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True)

# Balkenvergleich
kategorien = ["Parameter\n(×100)", "MAE (×10⁵)", "Zeit (s)"]
werte_lstm = [m_lstm.count_params()/100, mae_lstm*1e5, zeit_lstm]
werte_gru  = [m_gru.count_params()/100,  mae_gru*1e5,  zeit_gru]
x_pos   = np.arange(len(kategorien))
breite  = 0.35
axes[2].bar(x_pos - breite/2, werte_lstm, breite, label="LSTM", color="#3498db")
axes[2].bar(x_pos + breite/2, werte_gru,  breite, label="GRU",  color="#e67e22")
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(kategorien)
axes[2].set_title("GRU vs. LSTM: Direkter Vergleich")
axes[2].legend()
axes[2].grid(True, axis="y", alpha=0.3)

plt.suptitle("GRU vs. LSTM: Trainingsgeschwindigkeit, Genauigkeit, Parameter",
             fontsize=12)
plt.tight_layout()
plt.savefig("F9_3_gru_vs_lstm.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F9_3_gru_vs_lstm.png")
